# L4 37 — matched-layout adapter comparison

Compares the original and context-aware Qwen 3B links when readable text-token embeddings and continuous representations occupy the exact same receiver position after an identical prompt.

The exact-token oracle must have zero divergence by construction. This isolates sender-state mapping quality from the receiver-layout confound. No weights are updated.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (receiver secrets are optional)')

In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = 'a4df08356905c34a33e8f85ef742a3ed6980d156'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)

In [ ]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
ROOT = pathlib.Path('/content/drive/MyDrive/rival-arena-l4')
SOURCE_DIR = ROOT/'faithful-qwen3b-t4-001'
CONTEXT_DIR = ROOT/'contextual-qwen3b-t4-001'
MATCHES = SOURCE_DIR/'arena_ipd_confirmatory_v3/matches.jsonl'
for required in (SOURCE_DIR/'faithful_link.pt', CONTEXT_DIR/'faithful_link.pt', MATCHES):
    assert required.exists(), f'Missing required Drive artifact: {required}'
print('Comparing:', SOURCE_DIR.name, 'and', CONTEXT_DIR.name)

In [ ]:
for job_dir in (SOURCE_DIR, CONTEXT_DIR):
    command = [
        'python', 'scripts/diagnose_arena_fidelity.py',
        '--model', MODEL,
        '--job-dir', job_dir,
        '--job-id', job_dir.name,
        '--matches', MATCHES,
        '--samples', '256',
        '--seed', '20260727',
        '--layout', 'matched',
    ]
    print('\n' + ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=WORK/'followup-representational', check=True, env=os.environ)

## Completion

Each job directory receives `arena_context_fidelity_matched_v2/`. Download both updated job folders or the two report directories. We will select the adapter only from this frozen comparison.